# পাঠ ১৮: ক্রিপ্টোগ্রাফিক রসিদ দিয়ে AI এজেন্টদের সুরক্ষা প্রদান

## হাতে কলমে নোটবুক

এই নোটবুক চারটি অনুশীলন ব্যাখ্যা করে:

১. একটি এজেন্ট টুল কলের জন্য আপনার প্রথম রসিদ **স্বাক্ষর করুন** এবং এটি যাচাই করুন।
২. রসিদে **ছলনা করুন** এবং যাচাইকরণের ব্যর্থতা পরীক্ষা করুন।
৩. একটি তিন-রসিদ চেইন **তৈরি করুন** এবং চেইনের অখণ্ডতা নিশ্চিত করুন।
৪. একটি Microsoft Agent Framework টুল কল **মুড়িয়ে দিন** যাতে প্রতিটি অ্যাকশন একটি রসিদ ইমিট করে।

সমস্ত ক্রিপ্টোগ্রাফিক প্রিমিটিভগুলি ভালোভাবে রক্ষণাবেক্ষণ করা লাইব্রেরি থেকে আমদানি করা হয়েছে (`pynacl` Ed25519 এর জন্য, `jcs` RFC 8785 মূল JSON এর জন্য, `hashlib` পাইথন স্ট্যান্ডার্ড লাইব্রেরি থেকে SHA-256 এর জন্য)। রসিদ লজিক নিজেই সাধারণ পাইথন যা আপনি পড়তে এবং সংশোধন করতে পারেন।

সেলগুলি ধারাবাহিকভাবে চালান। প্রতিটি অংশ সংক্ষিপ্ত এবং স্বনির্ভর।


## সেটআপ

দুটি ডিপেনডেন্সি ইনস্টল করুন। উভয়ই অনুমোদনমূলক লাইসেন্স আছে (Apache-2.0 / MIT)।


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## সাহায্যকারী ইউটিলিটিসমূহ

এই দুইটি সাহায্যকারী বেস64url এনকোডিং (প্যাডিং ব্যতীত) এবং যেকোনো অবজেক্টের SHA-256 হ্যাশিং পরিচালনা করে। এগুলো নোটবুকের বাকিটাকে রসিদ লজিকেই কেন্দ্রীভূত রাখে।


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## বিভাগ ১: আপনার প্রথম রসিদে স্বাক্ষর করুন

কল্পনা করুন আমাদের **Contoso Travel** এজেন্ট কাস্টমারের জন্য সিডনি থেকে লস অ্যাঞ্জেলেসের ফ্লাইট দেখতে পেয়েছে। আমরা এই টুল কল সম্পর্কে গেটওয়ের বিবৃতি রেকর্ড করতে চাই যাতে ভবিষ্যতের অডিটর আলাদা পinned ইস্যুয়ার কী দ্বারা এটি যাচাই করতে পারে।

### ধাপ ১.১: একটি স্বাক্ষর কী তৈরি করুন

প্রোডাকশনে, এজেন্টের স্বাক্ষর কী হার্ডওয়্যার সিকিউরিটি মডিউল (HSM), Azure Key Vault, বা অনুরূপ সুরক্ষিত স্টোরে থাকবে। এই পাঠে আমরা মেমরিতে একটি নতুন কী তৈরি করব।


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### ধাপ ১.২: রসিদ পে-লোড তৈরি করুন

পে-লোডে সব কিছু থাকে যা আমরা চাই রসিদ দ্বারা প্রমাণিত হোক: কে কাজ করেছিল, কোন সরঞ্জামটি ব্যবহার করেছিল, কী আর্গুমেন্ট নিয়ে, কী ফলাফল এসেছে, কোন নীতিমালা অনুসারে, এবং কখন। আমরা আর্গুমেন্ট ও ফলাফলগুলি সরাসরি অন্তর্ভুক্ত না করে তাদের হ্যাশ করি যাতে রসিদ সংবেদনশীল বিষয়বস্তু ফাঁস না করে।


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### ধাপ ১.৩: রসিদে সই করুন এবং একত্রিত করুন

তিনটি ধাপ:

১। JCS ব্যবহার করে পে-লোডটি ক্যানোনিকালাইজ করুন যাতে একই লগিক্যাল রসিদ তৈরি করা দুটি ইমপ্লিমেন্টেশন সাবলীল বিভিন্ন বাইটের পরিবর্তে সম্পূর্ণ একই বাইট তৈরি করে।
২। Ed25519 প্রাইভেট কী দিয়ে সরাসরি ক্যানোনিকাল বাইটগুলিতে সই করুন। PureEdDSA অভ্যন্তরীণভাবে মেসেজ হ্যাশ করে, তাই অতিরিক্ত প্রি-হ্যাশ করলে প্রোটোকল পরিবর্তিত হবে।

এরপর সিগনেচারটি মূল পে-লোডের সাথে লাগিয়ে চূড়ান্ত রসিদ তৈরি করা হয়।


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### ধাপ ১.৪: রসিদ যাচাই করুন

যাচাই প্রক্রিয়াটি উল্টে দেয়। আমরা স্বাক্ষর অপসারণ করি, canonical বাইটগুলি পুনর্গণনা করি, এবং ইস্যুকর্তার পাবলিক কী এর বিপরীতে স্বাক্ষর যাচাই করি।

যাচাই অফলাইন হয়: কোনো সার্ভিস কল করতে হয় না এবং কোন নেটওয়ার্ক অ্যাক্সেসও দরকার হয় না। এর জন্য রসিদ ছাড়া আরেকটি জিনিস দরকার: ইস্যুকর্তার পাবলিক কী, যা এমন একটি চ্যানেলের মাধ্যমে প্রাপ্ত হয় যা নিরীক্ষক ইতিমধ্যেই বিশ্বাস করে, যেমন ইস্যুকর্তা প্রকাশিত JWK সেট বা আগাম পিন করা মান। রসিদের ভিতরে `public_key` ক্ষেত্রটি সেই ভূমিকা পালন করতে পারে না, কারণ রসিদ যিনি লিখেন তিনি সেই ক্ষেত্রটিও লেখেন। অধ্যায় ২ এই প্রতারণা প্রতিরোধ করে।


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


আপনি দেখতে পাবেন `Receipt is valid: True`, `Pre-hashed receipt valid: False` এবং `Malformed receipts rejected: True`। পজিটিভ কেসটি প্রমাণ করে যে ডিরেক্ট-JCS সাইনিং পথ কাজ করছে; নেগেটিভ কন্ট্রোলটি সিগনেচার-স্কোপ নিয়মকে কার্যকর করে তোলে, কেবল প্রজ হিসেবে ছেড়ে যাওয়ার পরিবর্তে; শেষ লাইনটি ভুল ইনপুটে `False` রিটার্ন করছে, যা একটি ব্যতিক্রম উত্তেজিত করার পরিবর্তে ঘটছে। 


## বিভাগ ২: রশিদে ছিনতাই করুন

রশিদের পুরো উদ্দেশ্য হল এগুলো ছিনতাই-বিরোধী হওয়া। এটি আমরা প্রমাণ করব।

আমরা রশিদের ঠিক একটি অক্ষর পরিবর্তন করব এবং যাচাই অযোগ্য হওয়া দেখব।


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### কি হলো ঠিক এখন?

যখন আমরা `policy_id` পরিবর্তন করেছিলাম, টেক্সটের canonical bytes পরিবর্তিত হলো। মূল canonical bytes এর উপরে থাকা স্বাক্ষরটি এখন আর মেলে না। যাচাই (verification) সঠিকভাবে `False` ফেরত দেয়।

কোনও স্বাক্ষরযুক্ত ক্ষেত্র, অর্থাৎ `signature` অবজেক্টের বাইরে যেকোনো কিছু পরিবর্তন করা এবং তবুও রসিদটি ইস্যুকৃত পাবলিক কী দিয়ে যাচাই করা সম্ভব নয়, যদি না আক্রমণকারী ইস্যুকর্তার প্রাইভেট কী ধারণ করেন। `signature` অবজেক্ট নিজে স্বাক্ষরিত নয়: `sig` হল স্বাক্ষর, এবং `verify_receipt` `public_key`-কে উপেক্ষা করে, তাই শুধুমাত্র ওই ক্ষেত্র সম্পাদনা করলেই কিছু পরিবর্তন হয় না। যতক্ষণ পর্যন্ত প্রাইভেট কী কী ভল্টে থাকে এবং যাচাইকারীরা ইস্যুকর্তার স্বাধীনভাবে প্রকাশিত কী ব্যবহার করে, স্বাক্ষরযুক্ত ক্ষেত্রের পরিবর্তন লুকানো অসম্ভব। এই শর্তটি গুরুত্বপূর্ণ: পরবর্তী সেলটি দেখায় পরিবর্তিত রসিদ যা অন্য কী দিয়ে সম্পূর্ণভাবে যাচাই হয়।

নিজে চেষ্টা করে দেখুন: উপরের সেলে `tool_name` বা `agent_id` বা `timestamp` সম্পাদনা করুন এবং পুনরায় চালান। স্বাক্ষরিত ক্ষেত্রের প্রতিটি পরিবর্তন অবৈধ রসিদ তৈরি করে।


### স্বাক্ষর যে ছানা ধরতে পারে না এমন ছানা পরিবর্তন

উপরের আক্রমণকারী পুনঃস্বাক্ষর করেনি। একটু বেশি সতর্ক কেউ করতো: রসিদ পরিবর্তন করতো,
নিজেরাই একটি কী তৈরি করে তাতে স্বাক্ষর করতো, এবং সেই কীটিকে
`public_key` ক্ষেত্রের মধ্যে রাখতো। ফলাফল হলো একটি রসিদ যার স্বাক্ষর তার বিষয়বস্তু সম্পূর্ণরূপে প্রকৃত।


এটা চালান এবং দেখুন কোন দুইটি চেক এর মধ্যে কোনটি এটি ধরে। 


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


আপনি দেখতে পাবেন `True` তারপর `False`। তিনটি `assert` লাইন এটিকে একটি
যাচাইকরণ করে: সেলটি একটি ত্রুটি তুলবে যদি জালিয়াতি কখনও ইস্যুকৃত কী-র বিরুদ্ধে
যাচাই হয়।

জালিয়াতি অভ্যন্তরীণভাবে সঙ্গতিপূর্ণ। এর স্বাক্ষর আসলেই এর
বিষয়বস্তু এবং এটি বহন করা কী-র সাথে মেলে, তাই একটি যাচাইকারী যে কী রসিদ থেকে পড়ে
তা বৈধ বলে রিপোর্ট করে। রসিদে কিছুই ভুল মনে হয় না, কারণ
এর কিছুই সঙ্গতিহীন নয়।

একমাত্র যা এটি ধরা পড়ায় তা হলো এমন একটি কী-র বিরুদ্ধে যাচাইকরণ যা আক্রমণকারী প্রভাবিত করতে পারেনি।
এজন্যই `verify_receipt` কীটিকে আর্গুমেন্ট হিসেবে গ্রহণ করে: একটি
ফাংশন যার স্বাক্ষর `verify_receipt(receipt)` cannot be safely লেখা যায় না,
কারণ এটি একমাত্র কী অ্যাক্সেস করতে পারত তা হলো যে কী আক্রমণকারী দিয়েছে।

একই রসিদটি `sample_receipts/04_resigned_receipt.json` হিসেবে সংরক্ষিত হয় যদি আপনি
নোটবুকের বাইরে এটি পরীক্ষা করতে চান।


## বিভাগ ৩: রসিদগুলি একসাথে লিঙ্ক করুন

একটি একক রসিদ একটি ক্রিয়াকলাপকে সুরক্ষা দেয়। বেশিরভাগ এজেন্ট বহু ক্রিয়াকলাপ গ্রহণ করে। পুরো সিকোয়েন্সটিকে পরিবর্তন-প্রমাণ করতে, আমরা প্রতিটি রসিদকে পূর্বের রসিদের সাথে লিঙ্ক করি পূর্বের রসিদটির হ্যাশ নতুন রসিদের পে’লোডে অন্তর্ভুক্ত করে।

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

যদি কেউ উপস্থাপিত সিকোয়েন্সের ভিতরে কোনো রসিদ অপসারণ করে বা পুনর্বিন্যাস করে, পরবর্তী লিঙ্কটি ভেঙে যায়। ট্রাঙ্কেশন বা সইকারী-নিয়ন্ত্রিত সাফিক্স পুনঃলিখন শনাক্ত করাও একটি পিন করা বর্তমান হেড, বাহ্যিক চেকপয়েন্ট, বা প্রমাণীকৃত জনসংখ্যার কমিটমেন্ট প্রয়োজন। 


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


এখন মধ্যবর্তী রসিদটি বদলে চেইনটি ভাঙুন এবং আবার যাচাই করুন। বদলানো রসিদ তার স্বাক্ষর পরীক্ষা ফেল করে, এবং পরবর্তী রসিদ তার চেইন-লিঙ্ক পরীক্ষা ফেল করে (কারণ এর `previous_receipt_hash` আর সংশোধিত মধ্যবর্তী রসিদের হ্যাশের সাথে মেলে না)।


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


রসিদ ০ এখনও যাচাই হয় (এটি পরিবর্তিত হয়নি এবং নির্ভর করার জন্য কোনও পূর্বসূরি নেই)। রসিদ ১ তার স্বাক্ষর যাচাইয়ে ব্যর্থ হয় কারণ আমরা `tool_args_hash` পরিবর্তন করেছি। রসিদ ২ তার চেইন-লিঙ্ক যাচাইয়ে ব্যর্থ হয় কারণ এর `previous_receipt_hash` মূল (এখন পরিবর্তিত) রসিদ ১ এর বিরুদ্ধে গণনা করা হয়েছিল।

এমনকি একজন আক্রমণকারী যদি পরিবর্তিত রসিদ ১ পুনরায় স্বাক্ষর করে, নতুন স্বাক্ষর ইস্যুকারের চাবির বিরুদ্ধে ব্যর্থ হয় যদি না তাদের কাছে ইস্যুকারের ব্যক্তিগত চাবি থাকে (তাদের নিজের চাবি দ্বারা পুনঃস্বাক্ষরও ব্যর্থ হয়, যেমন বিভাগ ২ দেখিয়েছে), এবং রসিদ ২ তে চেইন-লিঙ্ক মিল না থাকার কারণে এখনও হেরফের প্রকাশ পাবে। পরিবর্তন লুকাতে, আক্রমণকারীকে পরিবর্তনের বিন্দু থেকে শুরু করে প্রতিটি রসিদ পুনঃস্বাক্ষর করতে হবে, যার জন্য ইস্যুকারের ব্যক্তিগত চাবি থাকা আবশ্যক।


## ধারা ৪: রসিদ স্বাক্ষর সহ এজেন্ট টুল কল মোড়ানো

একটি বাস্তব স্থাপনার ক্ষেত্রে, আপনি চান না যে প্রতিটি এজেন্ট লেখক `make_receipt` কল করতে মনে রাখুক। আপনি চান প্রতিটি টুল চালানোর জন্য রসিদ স্বাক্ষর স্বয়ংক্রিয় হোক।

এখানে সবচেয়ে সহজ প্যাটার্ন: একটি র‍্যাপার ক্লাস যা যেকোনও কলযোগ্য টুল ফাংশন গ্রহণ করে এবং তার একটি রসিদ-উৎপাদন সংস্করণ ফিরে দেয়। এটি যেকোনো এজেন্ট ফ্রেমওয়ার্কের সাথে মানিয়ে যায়, যার মধ্যে রয়েছে মাইক্রোসফট এজেন্ট ফ্রেমওয়ার্ক (`agent_framework.foundry`)।

যদি আপনার কাছে মাইক্রোসফট ফাউন্ড্রি প্রকল্প না থাকে, তাহলে নিচের স্থানীয় মকটি প্যাটার্নটি এখনও প্রদর্শন করে।


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### মাইক্রোসফ্ট এজেন্ট ফ্রেমওয়ার্কের সাথে ইন্টিগ্রেশন

উপরোক্ত `ReceiptedTool` র‌্যাপারটি ফ্রেমওয়ার্ক-স্বাধীন। এটি মাইক্রোসফ্ট এজেন্ট ফ্রেমওয়ার্ক দিয়ে নির্মিত একটি এজেন্টের ভিতরে ব্যবহার করতে, র‌্যাপ করা ফাংশনটিকে একটি টুল হিসেবে রেজিস্টার করুন। একটি স্কেচ (আপনি মকটির পরিবর্তে আসল মাইক্রোসফ্ট ফাউন্ড্রি টুল রেজিস্ট্রেশন করবেন):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

এজেন্ট ফ্রেমওয়ার্ককে রশিদ সম্পর্কিত কোনো কিছু জানা দরকার নেই। রশিদ সাইন করা টুলের চারপাশে আবৃত, ফ্রেমওয়ার্কে যুক্ত নয়। এভাবেই আপনি এজেন্ট কোড পুনঃলিখন না করে বিদ্যমান এজেন্ট কোডে উৎপত্তি যুক্ত করেন।


## পুনরাবৃত্তি এবং স্ট্রেচ চ্যালেঞ্জ

আপনার কাছে রয়েছে:

- একটি Ed25519 কী পেয়ার তৈরি করা হয়েছে।
- একটি এজেন্ট টুল কলের জন্য রসিদ তৈরি এবং স্বাক্ষর করা হয়েছে।
- রসিদটি অফলাইন মোডে যাচাই করা হয়েছে শুধুমাত্র ইস্যুকৃত পাবলিক কী ব্যবহার করে, যা রসিদ থেকে স্বাধীনভাবে প্রাপ্ত।
- একটি রসিদে ছিনতাই করা হয়েছে এবং যাচাই ব্যর্থ হওয়া লক্ষ্য করা হয়েছে।
- তিনটি রসিদের একটি হ্যাশ-চেইনড সিকোয়েন্স নির্মাণ করা হয়েছে।
- চেইনের মধ্যভাগে ছিনতাই করা হয়েছে এবং উভয় স্বাক্ষর ব্যর্থতা এবং চেইন-লিঙ্ক ব্যর্থতা লক্ষ্য করা হয়েছে।
- একটি টুল ফাংশনের সাথে স্বয়ংক্রিয় রসিদ স্বাক্ষর মোড়ানো হয়েছে।

**স্ট্রেচ চ্যালেঞ্জ।** রসিদ স্কিমা একটি `request_id` ফিল্ড (বিতরণকৃত ট্রেসিংয়ের জন্য একটি UUID) দ্বারা বিস্তৃত করুন। `make_receipt` আপডেট করুন যাতে এটি অন্তর্ভুক্ত থাকে, এবং রসিদগুলি শেষ থেকে শেষ পর্যন্ত যাচাই করা হয় তা নিশ্চিত করুন। তারপর স্বাক্ষর করার পরে ফিল্ডটি পরিবর্তন করুন এবং যাচাই ব্যর্থ হওয়া নিশ্চিত করুন। এটি আপনাকে শেখায় কিভাবে প্রতিটি ক্যানোনিক্যাল এনকোডিংয়ের বাইট স্বাক্ষরে অবদান রাখে।

**গুরুত্বপূর্ণ সীমানা।** একটি পিন করা কী এবং চেইন প্রেক্ষাপটে, রসিদগুলি স্বাক্ষর স্বত্ব, বিষয়বস্তুর অখণ্ডতা এবং প্রদর্শিত পূর্বসূরীর সাথে একটি লিঙ্ক প্রমাণ করে। এটি কার্যনির্বাহীতা, উত্সের সত্যতা, বিশ্বাসযোগ্য সময়, সম্পূর্ণতা, নীতি মূল্যায়ন বা সঠিকতা প্রমাণ করে না। রসিদ হল একটি ভিত্তি। গভর্নেন্স হল সেই ব্যবস্থা যা আপনি এর উপর তৈরি করেন।

সেই সীমানা মাথায় রেখে আবার পাঠের README পড়ুন। দলের সবচেয়ে সাধারণ ভুল হল রসিদ থাকার মানেই "আমরা গভর্ন করা হচ্ছি" ধরে নেওয়া। তা নয়। রসিদ এজেন্টের আচরণ শুনানিবদ্ধ করে। এটি এটিকে সঠিক করে না।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
